# Phase 5 — Do the attention maps actually point at lesions?

**Question:** the paper says its attention "strongly concentrates on irregular mass margins, architectural distortions and suspicious microcalcifications", but only shows four pictures. Here every map is **scored against the radiologist-drawn lesion outlines (ROI masks) of CBIS-DDSM**, next to maps that know nothing about lesions (random, uniform, brightest tissue, breast centre).

- Test images: the **official CBIS-DDSM test split** only (patients the models never saw).
- Models: the four Phase 4 models, loaded from the Phase 4 notebook's Output. **No retraining.**
- Maps: CBAM spatial attention, Grad-CAM for the malignancy head and for the density head, plus an untrained network as a control.

**Before running**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet on**.
2. Add Input → **"CBIS-DDSM: Breast Cancer Image Dataset"** (by *awsaf49*) → Add.
3. Add Input → **Your Work → Notebooks → `notebookb86c6df37c`** (the Phase 4 notebook; its Output holds `checkpoints/`) → Add.
4. Run the first **two** cells (about 5 minutes). Cell 1 must list **4 checkpoints**. If cell 2 ends with `ALL CHECKS PASSED`, click **Save Version → Save & Run All (Commit)**. The full run takes about **20–30 minutes** and continues if you close the browser.
5. When it finishes, open the saved version → **Output** → download **`attention_results.zip`** and send it to Claude.


In [ ]:
# 1) Get the code, check the machine, find the Phase 4 checkpoints
import os, subprocess, sys, torch
REPO = "/kaggle/working/repo"
if os.path.exists(REPO):  # a folder left from an earlier run in this session: update it to the latest code
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
assert N_GPU > 0, "Turn on the GPU: Settings -> Accelerator -> GPU T4 x2, then run again."
os.environ["PYTHONPATH"] = f"{REPO}/src"
sys.path.insert(0, f"{REPO}/src")
from mammo.experiments.attention import find_checkpoints, VARIANTS
CKPTS = find_checkpoints("auto")
for v in VARIANTS:
    print(f"  {v:9s}", CKPTS.get(v, "MISSING"))
MISSING = [v for v in VARIANTS if v not in CKPTS]
if MISSING:
    print("\nNot all checkpoints found. Did you add the Phase 4 notebook (notebookb86c6df37c) as an input?\n"
          "If you continue anyway, cell 4 retrains the missing models (~8 min each).")


In [ ]:
# 2) Unit tests + 5-minute smoke run on 16 test images with untrained models. If this cell fails, stop and send Claude a screenshot.
!cd /kaggle/working/repo && python -m pytest -q tests 2>&1 | tail -3
!cd /kaggle/working/repo && python -m mammo.experiments.attention prepare --smoke \
  && python -m mammo.experiments.attention run --smoke \
  && python -m mammo.experiments.attention summarise --smoke \
  && echo "ALL CHECKS PASSED"


In [ ]:
# 3) Official test split: images + lesion masks in model space (~5 min).
#    Check the preview: the orange outlines should sit on a visible lesion (mass or calcification cluster).
!cd /kaggle/working/repo && python -m mammo.experiments.attention prepare
from IPython.display import Image, display
display(Image("/kaggle/working/results/attention/mask_preview.png"))


In [ ]:
# 4) Only if checkpoints are missing: retrain them on the official split, exactly as in Phase 4 (same seed).
import time
if MISSING:
    t0 = time.time()
    !cd /kaggle/working/repo && python -m mammo.experiments.cbis prepare
    for v in MISSING:
        !cd /kaggle/working/repo && python -m mammo.experiments.external train {v} --gpu 0
    print(f"retrained {MISSING} in {(time.time() - t0) / 60:.0f} min")
else:
    print("all four checkpoints found, nothing to retrain")


In [ ]:
# 5) Maps for every model + baselines, per-image metrics, random example gallery (~15 min)
!cd /kaggle/working/repo && python -m mammo.experiments.attention run --gpu 0
display(Image("/kaggle/working/results/attention/gallery.png"))


In [ ]:
# 6) Statistics (patient bootstrap), paper claims, chart
!cd /kaggle/working/repo && python -m mammo.experiments.attention summarise
display(Image("/kaggle/working/results/attention/localisation_chart.png"))


In [ ]:
# 7) Pack the results for Claude -> download attention_results.zip from Output (/kaggle/working)
!cd /kaggle/working && rm -f attention_results.zip && zip -qr attention_results.zip results/attention && ls -lh attention_results.zip
